# M6 Verification: Structural Planning Evidence

This notebook verifies M6a, the deterministic T1 planning-artifact measure. M6b textual content extraction remains a separate gated task and is not executed here.

In [ ]:
from pathlib import Path
import json
import sys

import pandas as pd
import plotly.express as px
from IPython.display import Image, display

ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / 'paper_v9').is_dir():
    ROOT = ROOT.parent
METRICS = ROOT / 'paper_v9' / 'data' / 'metrics'
FIGURES = ROOT / 'paper_v9' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(ROOT))
planning = pd.read_csv(METRICS / 'm6a_structural_planning.csv', dtype={'Semestre': str})
m6b_records = json.loads((METRICS / 'm6b_llm_planning_content.json').read_text(encoding='utf-8'))
m6b_metadata = json.loads((METRICS / 'm6_llm_planning_content.metadata.json').read_text(encoding='utf-8'))
metadata = json.loads((METRICS / 'm6_structural_planning.metadata.json').read_text(encoding='utf-8'))
legacy = pd.read_csv(ROOT / 'paper_v8' / 'data' / 'm6_t1_planning_quality.csv', dtype={'Semestre': str})
signals = pd.read_csv(ROOT / 'paper_v4' / 'advanced_metrics' / 'outputs' / 'team_level_signals.csv', dtype={'Semestre': str})

## 1. Verification contract

M6 belongs to RQ3. M6a measures observable T1 artifact presence and scope at `team_semester` grain. It does not claim semantic planning quality and does not call an LLM.

In [ ]:
paper_text = (ROOT / 'paper_v8' / 'latex_code' / 'main.tex').read_text(encoding='utf-8')
m6_position = paper_text.index(r'\textbf{M6 --')
rq3_position = paper_text.index(r'\subsubsection{RQ3:')
assert rq3_position < m6_position
assert metadata['rq'] == 'RQ3'
assert metadata['unit_of_analysis'] == 'team_semester'
assert metadata['checkpoint'] == 'T1'
assert metadata['llm_calls_required'] is False
assert metadata['inference'] == 'exploratory_descriptive'
assert metadata['absence_policy'] == 'report_separately_not_as_score_floor'
assert len(planning) == 14
assert len(m6b_records) == 14
assert m6b_metadata['coverage'] == {'team_semester_n': 14, 'success_n': 9, 'unavailable_not_measured_n': 5}
assert m6b_metadata['gate_status'] == 'approved_human_review'
print('M6a/M6b RQ3, grain, checkpoint, coverage, and approved-review contracts: PASS')

## 2. V8 to V9 traceability

The V8 score is retained only as an audit reference. M6a does not reuse or reinterpret the opaque score as structural quality.

In [ ]:
traceability = pd.DataFrame([
    {'v8_recommendation': 'Preserve the T1 planning metric as a team-semester measure.', 'v9_decision': 'Publish one deterministic M6a row per team-semester from planning_metrics.parquet.', 'status': 'applied', 'evidence': 'm6a_structural_planning.csv', 'limitation_or_approval': 'Structural observation, not semantic quality.'},
    {'v8_recommendation': 'Separate presence and scope from textual planning content.', 'v9_decision': 'M6a contains structural fields; approved M6b publishes separate validated textual categories.', 'status': 'applied', 'evidence': 'M6a/M6b outputs and M9 structured associations', 'limitation_or_approval': 'No composite M6b score; five cases remain unavailable.'},
    {'v8_recommendation': 'Keep missing planning evidence distinct from a low score.', 'v9_decision': 'M6a preserves availability and M6b records unavailable_not_measured for no T1 subjects.', 'status': 'applied', 'evidence': 'M6a metadata and M6b JSON', 'limitation_or_approval': 'Repository absence does not prove no off-repository planning.'},
    {'v8_recommendation': 'Avoid treating the legacy LLM score alone as architectural quality.', 'v9_decision': 'Legacy score is audit-only; M6b uses structured categories and literal quotes, not a numeric score.', 'status': 'applied', 'evidence': 'M6a/M6b schemas and approved protocol', 'limitation_or_approval': 'M6b remains exploratory.'},
])
assert set(traceability['status']) == {'applied'}
display(traceability)
display(pd.DataFrame([{'legacy_rows': len(legacy), 'legacy_scored_n': int(legacy['t1_planning_score'].notna().sum()), 'legacy_missing_n': int(legacy['t1_planning_score'].isna().sum()), 'm6b_success_n': sum(record['status'] == 'success' for record in m6b_records), 'm6b_unavailable_n': sum(record['status'] == 'unavailable_not_measured' for record in m6b_records)}]))

In [ ]:
required_columns = {'ID_Equipe', 'Semestre', 'pi_available', 'pi_unavailable_reason', 'pi_file_count_t1', 'pi_line_delta_t1', 'planning_artifact_present_t1', 'planning_scope_log1p_t1', 'measurement_status', 'analysis_level'}
assert not (required_columns - set(planning.columns))
assert planning.duplicated(['ID_Equipe', 'Semestre']).sum() == 0
assert planning['pi_available'].notna().all()
assert planning['pi_file_count_t1'].ge(0).all()
assert planning['pi_line_delta_t1'].ge(0).all()
assert planning['planning_scope_log1p_t1'].ge(0).all()
assert planning['planning_artifact_present_t1'].equals(planning['pi_file_count_t1'].gt(0))
assert planning['measurement_status'].eq('available').all()
assert planning['analysis_level'].eq('team_semester').all()
assert 't1_planning_score' not in planning.columns
print('M6a schema, ranges, keys, absence handling, and no-score-floor invariant: PASS')

## 3. Article-ready artifacts

Figures consume the official M6a CSV. Presence is shown separately from scope so zero artifacts are not visually conflated with missing data.

In [ ]:
plot_data = planning.copy()
plot_data['team_semester'] = plot_data['Semestre'] + '/' + plot_data['ID_Equipe']
scope_figure = px.bar(plot_data, x='team_semester', y='planning_scope_log1p_t1', color='Semestre', title='M6a T1 planning artifact scope')
scope_figure.update_yaxes(title='log(1 + T1 planning line delta)')
presence = planning.groupby('Semestre', as_index=False).agg(team_semester_n=('ID_Equipe', 'size'), planning_artifact_present_n=('planning_artifact_present_t1', 'sum'))
presence['planning_artifact_absent_n'] = presence['team_semester_n'] - presence['planning_artifact_present_n']
presence_long = presence.melt(id_vars=['Semestre', 'team_semester_n'], value_vars=['planning_artifact_present_n', 'planning_artifact_absent_n'], var_name='evidence_state', value_name='team_semester_n_state')
presence_figure = px.bar(presence_long, x='Semestre', y='team_semester_n_state', color='evidence_state', barmode='stack', title='M6a T1 planning artifact presence')
for figure, stem in ((scope_figure, 'm6a_planning_scope_t1'), (presence_figure, 'm6a_planning_presence_t1')):
    figure.write_html(METRICS / f'{stem}.html', include_plotlyjs='cdn')
    for extension in ('pdf', 'svg', 'png'):
        figure.write_image(FIGURES / f'{stem}.{extension}', scale=2 if extension == 'png' else 1)
for stem in ('m6a_planning_scope_t1', 'm6a_planning_presence_t1'):
    assert all((FIGURES / f'{stem}.{extension}').is_file() and (FIGURES / f'{stem}.{extension}').stat().st_size > 0 for extension in ('pdf', 'svg', 'png'))
    assert (METRICS / f'{stem}.html').is_file() and (METRICS / f'{stem}.html').stat().st_size > 0
print('M6a HTML, PDF, SVG, and PNG figures generated: PASS')

In [ ]:
display(planning.sort_values(['Semestre', 'ID_Equipe'])[['ID_Equipe', 'Semestre', 'planning_artifact_present_t1', 'pi_file_count_t1', 'pi_line_delta_t1', 'planning_scope_log1p_t1', 'measurement_status']])
display(presence)
display(Image(filename=str(FIGURES / 'm6a_planning_scope_t1.png'), width=900))
display(Image(filename=str(FIGURES / 'm6a_planning_presence_t1.png'), width=900))
print('M6a artifact demo: official table and figures displayed')

## Preliminary RQ3 analysis

M6a contributes an exploratory structural baseline for RQ3 by distinguishing whether T1 planning artifacts were observed and how much line-level scope they had. It does not measure architectural quality, planning intent, effort, project success, or causality. M6b content extraction is `unavailable_pending_structured_reprocessing` until its protocol, prompt, cost, and sample receive explicit approval.

In [ ]:
summary = planning.groupby('Semestre', as_index=False).agg(team_semester_n=('ID_Equipe', 'size'), planning_artifact_present_n=('planning_artifact_present_t1', 'sum'), median_planning_scope_log1p_t1=('planning_scope_log1p_t1', 'median'), total_planning_line_delta_t1=('pi_line_delta_t1', 'sum'))
summary['analysis_level'] = metadata['unit_of_analysis']
summary['inference'] = metadata['inference']
summary['m6b_status'] = 'approved_human_review'
assert summary['analysis_level'].eq('team_semester').all()
assert summary['inference'].eq('exploratory_descriptive').all()
display(summary)
print('Preliminary RQ3 reading: M6a provides structural T1 evidence; approved M6b fields remain exploratory and non-composite, and no causal claim is supported.')